# 07 Section4-Retrain Seed-Ensemble Model Optimization

This notebook trains from scratch. It does not read existing submission files and does not read saved prediction files.

Baseline path mirrors 04 notebook Section 4:

- feature policy: `base_922 + labelwise additive top-k`
- model: `mix_lgbm_catboost`
- base models: LGBM + CatBoost
- internal blend: probability mean
- Optuna/refit/seed/early stopping: same as 04 Section 4

The only exported candidate is the seed ensemble of this same recipe.
Q2/Q3 XGB, S4 CatBoost-heavy, and conservative blend variants are removed
from this notebook path.


## 1. Config / Allowed Input Path Check

In [ ]:

from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedShuffleSplit

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*does not have valid feature names.*")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)

ROW_KEY_COLS = ["subject_id", "lifelog_date"]
LABELS = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

SECTION07_MODE = "full"  # "smoke" -> Q2/Q3/S4, fewer trials
SECTION07_TRIALS = 50
SECTION07_SMOKE_TRIALS = 3
SECTION07_RANDOM_STATE = 42
SECTION07_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")

MODEL_CPU_THREADS = 6
OPTUNA_N_JOBS = 1
OPTUNA_SHOW_PROGRESS_BAR = True
EARLY_STOPPING_ROUNDS = 80
CLIP_EPS = 1e-6

MODEL_NAME = "mix_lgbm_catboost"
NORMALIZATION = "none"
FEATURE_SOURCE = "saved_anchor_feature_list_922"
EXPORT_FEATURE_SOURCE = "section11_base922_plus_labelwise_additive_policy"
BASELINE_INTERNAL_BLEND_METHOD = "probability_mean"
CANDIDATE_BLEND_METHOD = "logit_mean"

SECTION11_EXPORT_ADDITIVE_TOPK_BY_LABEL = {
    "Q1": 20,
    "Q2": 40,
    "Q3": 40,
    "S1": 0,
    "S2": 10,
    "S3": 80,
    "S4": 0,
}

SECTION07_V5_VALIDATIONS = ["public_start_tail", "subject_time_tail_25", "subject_time_tail_35"]
SECTION07_V5_ENABLE_PROBE = True
SECTION07_V5_REQUIRE_ACCEPTANCE_FOR_EXPORT = True
SECTION07_V5_PROBE_N_ESTIMATORS = 350
SECTION07_V5_CHANGE_TOPK_BY_LABEL = {
    "Q1": 10,
    "Q2": 30,
    "Q3": 40,
    "S1": 0,
    "S2": 10,
    "S3": 30,
    "S4": 10,
}
SECTION07_V5_FREQ_TOPK_BY_LABEL = {
    "Q1": 0,
    "Q2": 10,
    "Q3": 20,
    "S1": 0,
    "S2": 0,
    "S3": 10,
    "S4": 0,
}

SECTION07_SEED_ENSEMBLE_OFFSETS = [0, 20000, 40000]

if SECTION07_MODE == "smoke":
    LABELS_TO_RUN = ["Q2", "Q3", "S4"]
    SECTION07_TRIALS = SECTION07_SMOKE_TRIALS
    SECTION07_SEED_ENSEMBLE_OFFSETS = [0, 20000]
else:
    LABELS_TO_RUN = LABELS


def find_data_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for candidate in [base, base / "Y2025LifeLogDB"]:
            if (candidate / "ch2026_metrics_train.csv").exists() and (candidate / "ch2026_submission_sample.csv").exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not find Y2025LifeLogDB root from cwd={cwd}")


DATA_ROOT = find_data_root()
EXP_ROOT = DATA_ROOT / "experiments" / "260519_recovered_feature_model_v1"
SOURCE_OUT_DIR = EXP_ROOT / "data" / "raw_reverse_legacy_xgb"
OUT_DIR = EXP_ROOT / "data" / "section07_model_optuna_seed_ensemble"
REPORT_DIR = EXP_ROOT / "reports"
SUBMISSION_DIR = DATA_ROOT / "submission" / SECTION07_TIMESTAMP

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "train_labels": DATA_ROOT / "ch2026_metrics_train.csv",
    "sample_submission": DATA_ROOT / "ch2026_submission_sample.csv",
    "features_train": SOURCE_OUT_DIR / "features_train.parquet",
    "features_test": SOURCE_OUT_DIR / "features_test.parquet",
    "entropy_train": DATA_ROOT / "experiments/260520_timing_entropy_features_v1/data/timing_entropy_features_train.parquet",
    "entropy_test": DATA_ROOT / "experiments/260520_timing_entropy_features_v1/data/timing_entropy_features_test.parquet",
    "anchor_params": SOURCE_OUT_DIR / "mix_lgbm_catboost_none_full_legacy_entropy_hightrial_params.json",
    "section11_top_features": OUT_DIR.parent / "clean_anchor_exact_04" / "section11_feature_selection_probe" / "section11_top_features.csv",
}


def is_forbidden_input_path(path: Path) -> bool:
    norm = str(path.resolve()).replace("\\", "/").lower()
    name = path.name.lower()
    if "/submission/" in norm:
        return True
    if "test_predictions" in name or "prediction" in name and name.endswith((".csv", ".parquet")):
        return True
    return False


path_rows = []
for name, path in PATHS.items():
    forbidden = is_forbidden_input_path(path)
    path_rows.append({"name": name, "path": str(path), "exists": path.exists(), "forbidden_input": forbidden})
allowed_input_audit = pd.DataFrame(path_rows)
allowed_input_audit.to_csv(OUT_DIR / "section07_allowed_input_audit.csv", index=False)
display(allowed_input_audit)
missing = allowed_input_audit[~allowed_input_audit["exists"] & ~allowed_input_audit["name"].eq("section11_top_features")]
assert missing.empty, missing
forbidden = allowed_input_audit[allowed_input_audit["forbidden_input"]]
assert forbidden.empty, forbidden

print("SECTION07_SECTION4_RETRAIN_SEED_ENSEMBLE_OPTUNA")
print("MODE:", SECTION07_MODE)
print("LABELS_TO_RUN:", LABELS_TO_RUN)
print("SECTION07_TRIALS:", SECTION07_TRIALS)
print("MODEL_CPU_THREADS:", MODEL_CPU_THREADS)
print("OPTUNA_N_JOBS:", OPTUNA_N_JOBS)
print("MODEL_NAME:", MODEL_NAME)
print("FEATURE_SOURCE:", FEATURE_SOURCE)
print("EXPORT_FEATURE_SOURCE:", EXPORT_FEATURE_SOURCE)
print("BASELINE_INTERNAL_BLEND_METHOD:", BASELINE_INTERNAL_BLEND_METHOD)
print("CANDIDATE_BLEND_METHOD:", CANDIDATE_BLEND_METHOD)
print("SECTION07_V5_ENABLE_PROBE:", SECTION07_V5_ENABLE_PROBE)
print("SECTION07_V5_VALIDATIONS:", SECTION07_V5_VALIDATIONS)
print("SECTION07_V5_CHANGE_TOPK_BY_LABEL:", SECTION07_V5_CHANGE_TOPK_BY_LABEL)
print("SECTION07_V5_FREQ_TOPK_BY_LABEL:", SECTION07_V5_FREQ_TOPK_BY_LABEL)
print("SUBMISSION_DIR:", SUBMISSION_DIR)


## 2. Utilities

In [ ]:

def clean_for_json(obj):
    if isinstance(obj, dict):
        return {str(k): clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if not np.isfinite(val) else val
    if isinstance(obj, (np.ndarray, pd.Series)):
        return clean_for_json(obj.tolist())
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(clean_for_json(payload), indent=2, ensure_ascii=False), encoding="utf-8")


def feature_hash(cols: list[str]) -> str:
    return hashlib.sha256(json.dumps(list(cols), ensure_ascii=False).encode("utf-8")).hexdigest()


def coerce_key_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["subject_id"] = out["subject_id"].astype(str)
    out["lifelog_date"] = pd.to_datetime(out["lifelog_date"]).dt.strftime("%Y-%m-%d")
    return out


def merge_optional_features(base: pd.DataFrame, extra: pd.DataFrame) -> pd.DataFrame:
    extra_cols = [c for c in extra.columns if c not in ROW_KEY_COLS and c not in base.columns]
    return base.merge(extra[ROW_KEY_COLS + extra_cols], on=ROW_KEY_COLS, how="left")


def binary_logloss(y_true, pred) -> float:
    return float(log_loss(y_true, np.clip(np.asarray(pred, dtype="float64"), CLIP_EPS, 1 - CLIP_EPS), labels=[0, 1]))


def make_inner_es_split(y_values: np.ndarray, random_state: int):
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
    return next(splitter.split(np.zeros(len(y_values)), y_values))


def sanitize_X(X: pd.DataFrame) -> pd.DataFrame:
    return X.replace([np.inf, -np.inf], np.nan)


def logit(p):
    p = np.clip(np.asarray(p, dtype="float64"), CLIP_EPS, 1 - CLIP_EPS)
    return np.log(p / (1 - p))


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def logit_blend(preds: dict, weights=None):
    names = list(preds)
    if weights is None:
        weights = {name: 1.0 for name in names}
    total = float(sum(weights.get(name, 0.0) for name in names))
    if total <= 0:
        raise ValueError("blend weights must sum to a positive value")
    arr = np.vstack([logit(preds[name]) for name in names])
    w = np.asarray([weights.get(name, 0.0) / total for name in names], dtype="float64")
    return np.clip(sigmoid(np.average(arr, axis=0, weights=w)), CLIP_EPS, 1 - CLIP_EPS)


def prediction_stats(df: pd.DataFrame, candidate_name: str, reference=None) -> dict:
    row = {"candidate_name": candidate_name}
    for label in LABELS:
        vals = df[label].astype(float).to_numpy()
        row[f"{label}_mean"] = float(np.mean(vals))
        row[f"{label}_std"] = float(np.std(vals))
        row[f"{label}_min"] = float(np.min(vals))
        row[f"{label}_max"] = float(np.max(vals))
        if reference is not None and label in reference.columns:
            row[f"{label}_mean_abs_diff_vs_baseline"] = float(np.mean(np.abs(vals - reference[label].astype(float).to_numpy())))
    return row


## 3. Section 4 Feature Build Replay

In [ ]:

train_y = coerce_key_frame(pd.read_csv(PATHS["train_labels"]))
sample = coerce_key_frame(pd.read_csv(PATHS["sample_submission"]))
features_train_base = coerce_key_frame(pd.read_parquet(PATHS["features_train"]))
features_test_base = coerce_key_frame(pd.read_parquet(PATHS["features_test"]))
entropy_train = coerce_key_frame(pd.read_parquet(PATHS["entropy_train"]))
entropy_test = coerce_key_frame(pd.read_parquet(PATHS["entropy_test"]))

assert len(train_y) == 450, len(train_y)
assert len(sample) == 250, len(sample)
assert not train_y.duplicated(ROW_KEY_COLS).any()
assert not sample.duplicated(ROW_KEY_COLS).any()

features_train_aug = merge_optional_features(features_train_base, entropy_train)
features_test_aug = merge_optional_features(features_test_base, entropy_test)
features_train = train_y[ROW_KEY_COLS].merge(features_train_aug, on=ROW_KEY_COLS, how="left")
features_test = sample[ROW_KEY_COLS].merge(features_test_aug, on=ROW_KEY_COLS, how="left")
assert len(features_train) == len(train_y)
assert len(features_test) == len(sample)

anchor_payload = json.loads(PATHS["anchor_params"].read_text(encoding="utf-8"))
ANCHOR_FEATURES_BY_LABEL = {}
for label in LABELS:
    cols = [str(c) for c in anchor_payload.get(label, {}).get("features", []) if c not in ROW_KEY_COLS]
    missing_cols = [c for c in cols if c not in features_train.columns or c not in features_test.columns]
    if not cols or missing_cols:
        raise KeyError({"label": label, "n_features": len(cols), "missing": missing_cols[:20]})
    ANCHOR_FEATURES_BY_LABEL[label] = cols


def first_existing(columns: list[str], candidates: list[str]):
    colset = set(columns)
    for col in candidates:
        if col in colset:
            return col
    lower_map = {c.lower(): c for c in columns}
    for col in candidates:
        if col.lower() in lower_map:
            return lower_map[col.lower()]
    return None


def series_from(X: pd.DataFrame, col) -> pd.Series:
    if col is None or col not in X.columns:
        return pd.Series(np.nan, index=X.index, dtype="float64")
    return pd.to_numeric(X[col], errors="coerce").astype("float64")


def safe_div(num, den) -> pd.Series:
    num = pd.to_numeric(num, errors="coerce").astype("float64")
    den = pd.to_numeric(den, errors="coerce").astype("float64")
    out = num / den.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)


def sum_series(*parts) -> pd.Series:
    frame = pd.concat([pd.to_numeric(p, errors="coerce").astype("float64") for p in parts], axis=1)
    return frame.sum(axis=1, min_count=1)


SOURCE_ALIASES = {
    "bed_hour_diff": ["circ_bed_hour_diff"],
    "wake_hour_dev7": ["circ_wake_hour_dev7"],
    "evening_activity": ["evening_18_24_activity_event_rate_per_valid_hour", "evening_18_24_activity_time_mass_entropy"],
    "daytime_activity": ["daytime_06_18_activity_event_rate_per_valid_hour", "daytime_06_18_activity_time_mass_entropy", "wPedo_step_sum"],
    "presleep_screen": ["screen_presleep_3h_cnt", "presleep_21_24_screen_app_burst_count", "presleep_21_24_screen_app_event_rate_per_valid_hour"],
    "daily_screen": ["mScreenStatus_m_screen_use_sum", "mScreenStatus_m_screen_use_mean", "daytime_06_18_screen_app_event_rate_per_valid_hour"],
    "presleep_app": ["app_presleep2h_video", "app_presleep2h_other", "presleep_21_24_screen_app_event_time_entropy"],
    "daily_app": ["mUsageStats_m_usage_stats_list_count_sum", "mUsageStats_m_usage_stats_list_count_mean"],
    "evening_light": ["evening_18_24_light_env_time_mass_entropy", "bulk_daily_mLight_m_light_ema03_last"],
    "daytime_light": ["daytime_06_18_light_env_time_mass_entropy", "bulk_daily_mLight_m_light_median"],
    "sleep_light": ["sleep_fixed_24_33_light_env_time_mass_entropy", "bulk_sleep_mLight_m_light_iqr", "bulk_sleep_wLight_w_light_iqr"],
    "charging_std": ["mACStatus_m_charging_std"],
    "charging_mean": ["mACStatus_m_charging_mean"],
    "sleep_light_direction": ["bulk_sleep_wLight_w_light_direction_change_rate", "bulk_sleep_mLight_m_light_pct_change_abs_mean", "sleep_fixed_24_33_light_env_change_direction_entropy"],
    "sleep_activity_direction": ["bulk_sleep_mActivity_m_activity_direction_change_rate", "sleep_fixed_24_33_activity_change_direction_entropy"],
    "presleep_social_entropy": ["presleep_21_24_social_wifi_ble_time_mass_entropy", "wide_presleep_18_27_social_wifi_ble_time_mass_entropy"],
    "daytime_screen_burst": ["daytime_06_18_screen_app_burst_count"],
    "daily_hr_slope": ["bulk_daily_wHr_heart_rate_slope_per_hour"],
    "ambient_sleep_prev7": ["ambient_sleep_night_events_prev7_mean"],
    "ambient_sleep_current": ["ambient_night_events"],
    "sleep_screen_wake": ["screen_sleep_night_wake_cnt", "sleep_early_24_27_screen_app_time_mass_entropy"],
    "sleep_light_early": ["sleep_early_24_27_light_env_burst_count", "sleep_early_24_27_light_env_time_mass_entropy"],
    "sleep_light_mid": ["sleep_mid_27_30_light_env_burst_count", "sleep_mid_27_30_light_env_time_mass_entropy"],
    "sleep_light_late": ["sleep_late_30_33_light_env_burst_count", "sleep_late_30_33_light_env_time_mass_entropy"],
    "sleep_movement_burst": ["sleep_fixed_24_33_activity_burst_count", "sleep_fixed_24_33_activity_change_direction_entropy"],
    "sleep_disturbance_entropy": ["wide_sleep_20_36_sleep_disturbance_event_event_gap_entropy", "sleep_fixed_24_33_sleep_disturbance_event_event_gap_entropy"],
    "peak_disturbance_hour": ["wide_sleep_20_36_sleep_disturbance_event_peak_hour"],
    "sleep_event_gap_entropy": ["wide_sleep_20_36_sleep_disturbance_event_event_gap_entropy"],
    "sleep_disturbance_rate": ["wide_sleep_20_36_sleep_disturbance_event_event_rate_per_valid_hour", "sleep_fixed_24_33_sleep_disturbance_event_event_rate_per_valid_hour"],
    "presleep_mobility_change": ["presleep_21_24_activity_change_direction_entropy", "wide_presleep_18_27_activity_change_direction_entropy"],
    "presleep_hr_decreasing_slope": ["presleep_hr_decreasing_slope", "bulk_presleep_wHr_heart_rate_slope_per_hour"],
    "presleep_light": ["presleep_21_24_light_env_time_mass_entropy", "presleep_21_24_light_env_event_rate_per_valid_hour"],
    "sleep_wifi_count_std": ["sleep_wifi_count_std", "bulk_sleep_mWifi_m_wifi_list_count_std"],
    "sleep_light_diff": ["sleep_light_diff", "bulk_sleep_mLight_m_light_pct_change_abs_mean", "bulk_sleep_wLight_w_light_pct_change_abs_mean"],
    "sleep_activity_burst": ["sleep_fixed_24_33_activity_burst_count"],
    "sleep_screen_event_count": ["sleep_fixed_24_33_screen_app_burst_count", "sleep_fixed_24_33_screen_app_event_rate_per_valid_hour"],
    "sleep_duration": ["circ_sleep_duration_h", "sleep_duration_h", "sleep_duration"],
    "daytime_screen_count": ["daytime_06_18_screen_app_event_rate_per_valid_hour", "daytime_06_18_screen_app_burst_count"],
    "daytime_mobility_distance": ["daytime_06_18_mobility_gps_event_rate_per_valid_hour", "mGps_distance_sum"],
    "daytime_social_burst": ["daytime_06_18_social_wifi_ble_burst_count", "daytime_06_18_social_wifi_ble_event_rate_per_valid_hour"],
    "evening_screen_count": ["evening_18_24_screen_app_event_rate_per_valid_hour", "evening_18_24_screen_app_burst_count"],
    "evening_mobility_change": ["evening_18_24_activity_change_direction_entropy", "evening_18_24_mobility_gps_event_gap_entropy"],
    "evening_social_count": ["evening_18_24_social_wifi_ble_event_rate_per_valid_hour", "evening_18_24_social_wifi_ble_burst_count"],
    "evening_hr_decreasing_slope": ["evening_hr_decreasing_slope", "bulk_evening_wHr_heart_rate_slope_per_hour"],
    "hrv_rmssd_prev3": ["hrv_rmssd_prev3", "bulk_daily_wHr_rmssd_prev3_mean"],
    "hrv_rmssd_prev7": ["hrv_rmssd_prev7", "bulk_daily_wHr_rmssd_prev7_mean"],
    "hrv_pnn50_prev3": ["hrv_pnn50_prev3", "bulk_daily_wHr_pnn50_prev3_mean"],
    "hrv_pnn50_prev7": ["hrv_pnn50_prev7", "bulk_daily_wHr_pnn50_prev7_mean"],
    "hr_sleep_resting_prev3": ["hr_sleep_resting_prev3", "bulk_sleep_wHr_heart_rate_prev3_mean"],
    "hr_sleep_resting_prev7": ["hr_sleep_resting_prev7", "bulk_sleep_wHr_heart_rate_prev7_mean"],
    "hr_sleep_mean": ["bulk_sleep_wHr_heart_rate_mean", "hr_sleep_mean"],
    "hr_sleep_max": ["bulk_sleep_wHr_heart_rate_max", "hr_sleep_max"],
    "hr_sleep_min": ["bulk_sleep_wHr_heart_rate_min", "hr_sleep_min"],
    "daytime_mobility_direction_entropy": ["daytime_06_18_activity_change_direction_entropy", "daytime_06_18_mobility_gps_change_direction_entropy"],
    "sleep_light_pct_change": ["bulk_sleep_mLight_m_light_pct_change_abs_mean", "bulk_sleep_wLight_w_light_pct_change_abs_mean"],
    "sleep_screen_entropy_mid": ["sleep_mid_27_30_screen_app_event_gap_entropy", "sleep_mid_27_30_screen_app_time_mass_entropy"],
    "presleep_mobility_entropy": ["presleep_21_24_activity_event_gap_entropy", "wide_presleep_18_27_activity_event_gap_entropy"],
    "presleep_light_mean": ["presleep_21_24_light_env_time_mass_entropy", "bulk_presleep_mLight_m_light_mean"],
    "presleep_light_change": ["presleep_21_24_light_env_change_direction_entropy", "bulk_presleep_mLight_m_light_pct_change_abs_mean"],
    "daytime_light_entropy": ["daytime_06_18_light_env_time_mass_entropy", "daytime_06_18_light_env_event_time_entropy"],
    "daytime_light_mean": ["bulk_daily_mLight_m_light_mean", "daytime_06_18_light_env_event_rate_per_valid_hour"],
    "evening_light_mean": ["evening_18_24_light_env_event_rate_per_valid_hour", "bulk_evening_mLight_m_light_mean"],
    "evening_light_change": ["evening_18_24_light_env_change_direction_entropy", "bulk_evening_mLight_m_light_pct_change_abs_mean"],
    "sleep_screen_coverage": ["sleep_fixed_24_33_screen_app_coverage"],
    "sleep_light_slope": ["sleep_fixed_24_33_light_env_change_direction_entropy", "bulk_sleep_mLight_m_light_slope_per_hour"],
    "daily_app_pct_change_abs": ["bulk_daily_mUsageStats_m_usage_stats_list_count_pct_change_abs_mean", "daily_app_pct_change_abs"],
    "sleep_duration_prev3": ["sleep_duration_prev3", "circ_sleep_duration_h_prev3_mean"],
    "sleep_duration_prev7": ["sleep_duration_prev7", "circ_sleep_duration_h_prev7_mean"],
    "screen_sleep_night_wake_prev3": ["screen_sleep_night_wake_prev3", "screen_sleep_night_wake_cnt_prev3_mean"],
    "screen_sleep_night_wake_prev7": ["screen_sleep_night_wake_prev7", "screen_sleep_night_wake_cnt_prev7_mean"],
    "presleep_2h_screen_burst": ["presleep_22_24_screen_app_burst_count"],
    "presleep_3h_screen_burst": ["presleep_21_24_screen_app_burst_count"],
    "presleep_4h_screen_burst": ["wide_presleep_18_27_screen_app_burst_count", "presleep_20_24_screen_app_burst_count"],
    "presleep_2h_social_count": ["presleep_22_24_social_wifi_ble_event_rate_per_valid_hour", "presleep_22_24_social_wifi_ble_burst_count"],
    "presleep_3h_social_count": ["presleep_21_24_social_wifi_ble_event_rate_per_valid_hour", "presleep_21_24_social_wifi_ble_burst_count"],
    "presleep_4h_social_count": ["wide_presleep_18_27_social_wifi_ble_event_rate_per_valid_hour", "wide_presleep_18_27_social_wifi_ble_burst_count"],
    "presleep_2h_mobility_change": ["presleep_22_24_activity_change_direction_entropy"],
    "presleep_3h_mobility_change": ["presleep_21_24_activity_change_direction_entropy"],
    "presleep_4h_mobility_change": ["wide_presleep_18_27_activity_change_direction_entropy"],
    "presleep_2h_light_change": ["presleep_22_24_light_env_change_direction_entropy"],
    "presleep_3h_light_change": ["presleep_21_24_light_env_change_direction_entropy"],
    "presleep_4h_light_change": ["wide_presleep_18_27_light_env_change_direction_entropy"],
}


def resolve_sources(columns: list[str]) -> dict:
    return {name: first_existing(columns, candidates) for name, candidates in SOURCE_ALIASES.items()}


def calendar_features(meta_df: pd.DataFrame) -> dict:
    dates = pd.to_datetime(meta_df["lifelog_date"])
    return {
        "v4_cal_dayofweek": dates.dt.dayofweek.astype(float),
        "v4_cal_is_weekend": dates.dt.dayofweek.isin([5, 6]).astype(float),
        "v4_cal_day": dates.dt.day.astype(float),
        "v4_cal_doy_sin": pd.Series(np.sin(2 * np.pi * dates.dt.dayofyear / 366.0), index=meta_df.index),
        "v4_cal_doy_cos": pd.Series(np.cos(2 * np.pi * dates.dt.dayofyear / 366.0), index=meta_df.index),
        "v4_cal_dow_sin": pd.Series(np.sin(2 * np.pi * dates.dt.dayofweek / 7.0), index=meta_df.index),
        "v4_cal_dow_cos": pd.Series(np.cos(2 * np.pi * dates.dt.dayofweek / 7.0), index=meta_df.index),
        "v4_cal_month_sin": pd.Series(np.sin(2 * np.pi * dates.dt.month / 12.0), index=meta_df.index),
        "v4_cal_month_cos": pd.Series(np.cos(2 * np.pi * dates.dt.month / 12.0), index=meta_df.index),
    }


def additive_feature_block(X: pd.DataFrame, meta_df: pd.DataFrame, source_map: dict) -> pd.DataFrame:
    s = lambda name: series_from(X, source_map.get(name))
    cols = dict(calendar_features(meta_df))
    resolved_aliases = [name for name, col in source_map.items() if col]
    for name in resolved_aliases[:60]:
        cols[f"v4src_{name}"] = s(name)

    q2_day_load = sum_series(s("daytime_screen_count"), s("daytime_mobility_distance"), s("daily_app"), s("daytime_social_burst"))
    q2_evening_recovery = sum_series(-s("evening_screen_count"), -s("evening_mobility_change"), -s("evening_social_count"), s("evening_hr_decreasing_slope"))
    hr_sleep_range = s("hr_sleep_max") - s("hr_sleep_min")
    sleep_light_spikes = sum_series(s("sleep_light_early"), s("sleep_light_mid"), s("sleep_light_late"))

    direct = {
        "v4_q1_sleep_fragmentation_rate": safe_div(s("sleep_screen_event_count"), s("sleep_duration").clip(lower=1e-3)),
        "v4_q1_hrv_pnn50_short_vs_long": s("hrv_pnn50_prev3") - s("hrv_pnn50_prev7"),
        "v4_q1_sleep_environment_instability_raw": sum_series(s("sleep_wifi_count_std"), s("sleep_light_diff"), s("sleep_activity_burst")),
        "v4_q2_bed_hour_shift_abs": s("bed_hour_diff").abs(),
        "v4_q2_wake_hour_dev7_abs": s("wake_hour_dev7").abs(),
        "v4_q2_evening_activity_over_daytime": safe_div(s("evening_activity"), s("daytime_activity")),
        "v4_q2_presleep_screen_over_daily": safe_div(s("presleep_screen"), s("daily_screen")),
        "v4_q2_presleep_app_over_daily": safe_div(s("presleep_app"), s("daily_app")),
        "v4_q2_light_evening_over_day": safe_div(s("evening_light"), s("daytime_light")),
        "v4_q2_charging_std_over_mean": safe_div(s("charging_std"), s("charging_mean")),
        "v4_q2_sleep_light_over_daylight": safe_div(s("sleep_light"), s("daytime_light")),
        "v4_q2_daytime_load_composite_raw": q2_day_load,
        "v4_q2_evening_recovery_composite_raw": q2_evening_recovery,
        "v4_q2_fatigue_proxy_raw": q2_day_load - q2_evening_recovery,
        "v4_q2_sleep_debt_current_proxy": (7.0 - s("sleep_duration")).clip(lower=0),
        "v4_q3_sleep_light_instability": s("sleep_light_direction").abs(),
        "v4_q3_sleep_activity_instability": s("sleep_activity_direction").abs(),
        "v4_q3_presleep_social_entropy": s("presleep_social_entropy"),
        "v4_q3_daytime_screen_burst": s("daytime_screen_burst"),
        "v4_q3_daily_hr_slope_abs": s("daily_hr_slope").abs(),
        "v4_q3_sleep_ambient_prev7_delta": s("ambient_sleep_current") - s("ambient_sleep_prev7"),
        "v4_q3_presleep_screen_to_sleep_light_ratio": safe_div(s("presleep_screen"), s("sleep_light")),
        "v4_q3_autonomic_arousal_raw": sum_series(s("daily_hr_slope"), -s("hrv_rmssd_prev3"), hr_sleep_range),
        "v4_q3_daytime_restlessness_raw": sum_series(s("daytime_screen_burst"), s("daytime_mobility_direction_entropy")),
        "v4_q3_presleep_arousal_2h_raw": sum_series(s("presleep_2h_screen_burst"), s("presleep_2h_social_count")),
        "v4_q3_presleep_arousal_3h_raw": sum_series(s("presleep_3h_screen_burst"), s("presleep_3h_social_count")),
        "v4_q3_presleep_arousal_4h_raw": sum_series(s("presleep_4h_screen_burst"), s("presleep_4h_social_count")),
        "v4_q3_sleep_unrest_raw": sum_series(s("sleep_light_pct_change"), s("sleep_activity_direction"), s("sleep_light_direction")),
        "v4_s2_hrv_rmssd_slope_approx": (s("hrv_rmssd_prev3") - s("hrv_rmssd_prev7")) / 4.0,
        "v4_s2_hrv_rmssd_acceleration": s("hrv_rmssd_prev3") - s("hrv_rmssd_prev7"),
        "v4_s2_sleep_continuity_raw": -sum_series(s("sleep_screen_entropy_mid"), s("sleep_light_diff"), s("sleep_activity_burst")),
        "v4_s2_daytime_social_to_recovery_proxy_raw": -s("daytime_social_burst") * q2_evening_recovery,
        "v4_s3_hr_baseline_raw": sum_series(s("hr_sleep_resting_prev7"), s("hr_sleep_mean"), s("hr_sleep_max")),
        "v4_s3_presleep_restlessness_raw": sum_series(s("presleep_mobility_entropy"), s("evening_mobility_change")),
        "v4_s3_presleep_darkness_score_raw": -sum_series(s("presleep_light_mean"), s("presleep_light_change")),
        "v4_s4_sleep_screen_wake_proxy": s("sleep_screen_wake"),
        "v4_s4_sleep_light_spike_proxy": sleep_light_spikes,
        "v4_s4_sleep_movement_fragmentation_proxy": s("sleep_movement_burst"),
        "v4_s4_sleep_disturbance_entropy": s("sleep_disturbance_entropy"),
        "v4_s4_peak_disturbance_hour": s("peak_disturbance_hour"),
        "v4_s4_sleep_event_gap_entropy": s("sleep_event_gap_entropy"),
        "v4_s4_first_stable_sleep_block_proxy": 1.0 / (1.0 + s("sleep_disturbance_rate").clip(lower=0)),
        "v4_s4_daytime_light_entrainment_raw": -s("daytime_light_entropy") + s("daytime_light_mean"),
        "v4_s4_evening_light_winddown_raw": -sum_series(s("evening_light_mean"), s("evening_light_change")),
        "v4_s4_sleep_awakening_composite_raw": sum_series(s("sleep_screen_coverage"), s("sleep_light_slope"), s("sleep_activity_burst")),
        "v4_s4_daytime_instability_to_sleep_raw": sum_series(s("daytime_mobility_direction_entropy"), s("daytime_light_entropy"), s("daily_app_pct_change_abs")),
        "v4_trend_hrv_rmssd_short_vs_long": s("hrv_rmssd_prev3") - s("hrv_rmssd_prev7"),
        "v4_trend_hrv_pnn50_short_vs_long": s("hrv_pnn50_prev3") - s("hrv_pnn50_prev7"),
        "v4_trend_hr_resting_short_vs_long": s("hr_sleep_resting_prev3") - s("hr_sleep_resting_prev7"),
        "v4_trend_sleep_duration_short_vs_long": s("sleep_duration_prev3") - s("sleep_duration_prev7"),
        "v4_trend_ambient_sleep_short_vs_long": s("ambient_sleep_current") - s("ambient_sleep_prev7"),
        "v4_trend_screen_sleep_short_vs_long": s("screen_sleep_night_wake_prev3") - s("screen_sleep_night_wake_prev7"),
        "v4_presleep_2h_light_change": s("presleep_2h_light_change"),
        "v4_presleep_3h_light_change": s("presleep_3h_light_change"),
        "v4_presleep_4h_light_change": s("presleep_4h_light_change"),
        "v4_presleep_2h_mobility_change": s("presleep_2h_mobility_change"),
        "v4_presleep_3h_mobility_change": s("presleep_3h_mobility_change"),
        "v4_presleep_4h_mobility_change": s("presleep_4h_mobility_change"),
    }
    cols.update(direct)
    return pd.DataFrame(cols, index=X.index).replace([np.inf, -np.inf], np.nan).loc[:, lambda df: ~df.columns.duplicated()].copy()


SOURCE_MAP = resolve_sources(sorted((set(features_train.columns) & set(features_test.columns)) - set(ROW_KEY_COLS)))
ADDITIVE_TRAIN = additive_feature_block(features_train, train_y[ROW_KEY_COLS], SOURCE_MAP)
ADDITIVE_TEST = additive_feature_block(features_test, sample[ROW_KEY_COLS], SOURCE_MAP)
assert len(ADDITIVE_TRAIN) == len(train_y)
assert len(ADDITIVE_TEST) == len(sample)
assert list(ADDITIVE_TRAIN.columns) == list(ADDITIVE_TEST.columns)


V5_DOMAIN_TOKENS = {
    "activity_mobility": ["activity", "pedo", "gps", "mobility"],
    "screen_app": ["screen", "app", "usagestats"],
    "light_ambient": ["light", "ambient", "noise", "snoring"],
    "hr_hrv": ["hr_", "heart_rate", "hrv", "rmssd", "pnn50"],
    "sleep": ["sleep", "circ_bed", "circ_wake"],
    "social_wifi_ble": ["wifi", "ble", "social"],
}
V5_SUFFIXES = [
    ("_direction_change_rate", "direction_change_rate"),
    ("_pct_change_abs_mean", "pct_change_abs_mean"),
    ("_diff_abs_mean", "diff_abs_mean"),
    ("_slope_per_hour", "slope_per_hour"),
    ("_roll3_mean_last", "roll3_mean_last"),
    ("_roll3_std_mean", "roll3_std_mean"),
    ("_ema03_last", "ema03_last"),
    ("_prev3_mean", "prev3_mean"),
    ("_prev7_mean", "prev7_mean"),
    ("_prev3_std", "prev3_std"),
    ("_prev7_std", "prev7_std"),
    ("_lag1", "lag1"),
    ("_diff1", "diff1"),
    ("_first", "first"),
    ("_last", "last"),
    ("_median", "median"),
    ("_mean", "mean"),
    ("_std", "std"),
    ("_iqr", "iqr"),
]


def v5_domain_for_col(col: str):
    lower = col.lower()
    for domain, tokens in V5_DOMAIN_TOKENS.items():
        if any(token in lower for token in tokens):
            return domain
    return None


def v5_split_suffix(col: str):
    for suffix, stat in V5_SUFFIXES:
        if col.endswith(suffix):
            return col[: -len(suffix)], stat
    return None, None


def v5_safe_name(text: str) -> str:
    safe = "".join(ch if ch.isalnum() else "_" for ch in str(text))
    while "__" in safe:
        safe = safe.replace("__", "_")
    return safe.strip("_")[:90]


def v5_sequence_groups(columns: list[str]) -> list[dict]:
    colset = set(columns)
    groups = {}
    for col in columns:
        if col in ROW_KEY_COLS or col.startswith("v4_") or col.startswith("v5_"):
            continue
        domain = v5_domain_for_col(col)
        if domain is None:
            continue
        stem, stat = v5_split_suffix(col)
        if stem is None:
            continue
        row = groups.setdefault(stem, {"stem": stem, "domain": domain, "parts": {}})
        row["parts"][stat] = col
    for stem, row in groups.items():
        if stem in colset:
            row["parts"]["current"] = stem
    scored = []
    for row in groups.values():
        parts = row["parts"]
        score = len(parts)
        score += 2 * int("prev3_mean" in parts and "prev7_mean" in parts)
        score += 2 * int("first" in parts and "last" in parts)
        score += 1 * int("slope_per_hour" in parts)
        if score >= 2:
            row["score"] = score
            scored.append(row)
    return sorted(scored, key=lambda r: (-r["score"], r["domain"], r["stem"]))


def v5_change_feature_block(X: pd.DataFrame, groups: list[dict], max_groups: int = 180):
    cols, audit = {}, []
    for row in groups[:max_groups]:
        stem, domain, parts = row["stem"], row["domain"], row["parts"]
        prefix = f"v5chg_{domain}_{v5_safe_name(stem)}"
        made = []
        get = lambda stat: series_from(X, parts.get(stat))
        if "prev3_mean" in parts and "prev7_mean" in parts:
            cols[f"{prefix}_short_vs_long"] = get("prev3_mean") - get("prev7_mean")
            cols[f"{prefix}_short_long_ratio"] = safe_div(get("prev3_mean"), get("prev7_mean"))
            made += ["short_vs_long", "short_long_ratio"]
        if "prev3_std" in parts and "prev7_std" in parts:
            cols[f"{prefix}_volatility_ratio"] = safe_div(get("prev3_std"), get("prev7_std"))
            made.append("volatility_ratio")
        if "last" in parts and "first" in parts:
            cols[f"{prefix}_last_minus_first"] = get("last") - get("first")
            made.append("last_minus_first")
        if "current" in parts and "lag1" in parts:
            cols[f"{prefix}_current_minus_lag1"] = get("current") - get("lag1")
            made.append("current_minus_lag1")
        if "mean" in parts and "ema03_last" in parts:
            cols[f"{prefix}_mean_minus_ema03"] = get("mean") - get("ema03_last")
            made.append("mean_minus_ema03")
        if "last" in parts and "ema03_last" in parts:
            cols[f"{prefix}_last_minus_ema03"] = get("last") - get("ema03_last")
            made.append("last_minus_ema03")
        if "slope_per_hour" in parts:
            cols[f"{prefix}_slope_abs"] = get("slope_per_hour").abs()
            made.append("slope_abs")
        inst_parts = [get(stat).abs() for stat in ["diff_abs_mean", "pct_change_abs_mean", "direction_change_rate"] if stat in parts]
        if inst_parts:
            cols[f"{prefix}_instability_score"] = sum_series(*inst_parts)
            made.append("instability_score")
        for kind in made:
            audit.append({
                "block": "v5_change",
                "stem": stem,
                "domain": domain,
                "feature_kind": kind,
                "source_cols_json": json.dumps(parts, ensure_ascii=False),
                "action": "created",
            })
    out = pd.DataFrame(cols, index=X.index).replace([np.inf, -np.inf], np.nan)
    return out.loc[:, ~out.columns.duplicated()].copy(), audit


def v5_frequency_feature_block(X: pd.DataFrame, groups: list[dict], max_groups: int = 100):
    cols, audit = {}, []
    stat_order = ["first", "median", "mean", "roll3_mean_last", "ema03_last", "last", "iqr", "std", "slope_per_hour", "direction_change_rate"]
    used = 0
    for row in groups:
        stem, domain, parts = row["stem"], row["domain"], row["parts"]
        available = [stat for stat in stat_order if stat in parts]
        if len(available) < 4:
            audit.append({
                "block": "v5_frequency",
                "stem": stem,
                "domain": domain,
                "feature_kind": "",
                "source_cols_json": json.dumps(parts, ensure_ascii=False),
                "action": "skipped_insufficient_sequence_proxy",
            })
            continue
        if used >= max_groups:
            break
        used += 1
        frame = pd.concat([series_from(X, parts[stat]) for stat in available], axis=1)
        frame.columns = available
        row_mean = frame.mean(axis=1, skipna=True)
        centered = frame.sub(row_mean, axis=0)
        diff = frame.diff(axis=1)
        row_std = frame.std(axis=1, skipna=True).replace(0, np.nan)
        prefix = f"v5freq_{domain}_{v5_safe_name(stem)}"
        cols[f"{prefix}_spectral_energy_proxy"] = (centered ** 2).mean(axis=1, skipna=True)
        cols[f"{prefix}_oscillation_proxy"] = diff.abs().mean(axis=1, skipna=True)
        cols[f"{prefix}_high_freq_proxy"] = (diff ** 2).mean(axis=1, skipna=True)
        if "first" in parts and "last" in parts:
            cols[f"{prefix}_trend_to_variance_ratio"] = safe_div((series_from(X, parts["last"]) - series_from(X, parts["first"])).abs(), row_std)
        for kind in ["spectral_energy_proxy", "oscillation_proxy", "high_freq_proxy", "trend_to_variance_ratio"]:
            audit.append({
                "block": "v5_frequency",
                "stem": stem,
                "domain": domain,
                "feature_kind": kind,
                "source_cols_json": json.dumps({stat: parts[stat] for stat in available}, ensure_ascii=False),
                "action": "created" if kind != "trend_to_variance_ratio" or ("first" in parts and "last" in parts) else "skipped_missing_first_last",
            })
    out = pd.DataFrame(cols, index=X.index).replace([np.inf, -np.inf], np.nan)
    return out.loc[:, ~out.columns.duplicated()].copy(), audit


def feature_block_audit(train_block: pd.DataFrame, source_audit: list[dict], path: Path) -> pd.DataFrame:
    source_df = pd.DataFrame(source_audit)
    stat_rows = []
    for col in train_block.columns:
        s = pd.to_numeric(train_block[col], errors="coerce")
        stat_rows.append({
            "feature": col,
            "mean_missing_rate": float(s.isna().mean()),
            "n_unique": int(s.nunique(dropna=True)),
            "is_constant": bool(s.nunique(dropna=True) <= 1),
            "is_near_constant": bool(s.value_counts(dropna=True, normalize=True).iloc[0] >= 0.98) if s.notna().any() else True,
        })
    stat_df = pd.DataFrame(stat_rows)
    if stat_df.empty:
        out = source_df if len(source_df) else stat_df
        out.to_csv(path, index=False)
        return out
    if len(source_df) and "feature_kind" in source_df.columns:
        created = source_df[source_df["action"].eq("created")].copy()
        created["feature_kind"] = created["feature_kind"].fillna("")
        created["feature_prefix"] = created.apply(
            lambda r: f"v5chg_{r['domain']}_{v5_safe_name(r['stem'])}_{r['feature_kind']}"
            if r["block"] == "v5_change"
            else f"v5freq_{r['domain']}_{v5_safe_name(r['stem'])}_{r['feature_kind']}",
            axis=1,
        )
        stat_df["feature_prefix"] = stat_df["feature"].str.replace("_short_vs_long|_short_long_ratio|_volatility_ratio|_last_minus_first|_current_minus_lag1|_mean_minus_ema03|_last_minus_ema03|_slope_abs|_instability_score|_spectral_energy_proxy|_oscillation_proxy|_high_freq_proxy|_trend_to_variance_ratio", "", regex=True)
        merged = stat_df.merge(created.drop_duplicates("feature_prefix"), on="feature_prefix", how="left")
        skipped = source_df[~source_df["action"].eq("created")].copy()
        out = pd.concat([merged.drop(columns=["feature_prefix"], errors="ignore"), skipped], ignore_index=True, sort=False)
    else:
        out = stat_df
    out.to_csv(path, index=False)
    return out


base_columns_for_v5 = sorted((set(features_train.columns) & set(features_test.columns)) - set(ROW_KEY_COLS))
V5_GROUPS = v5_sequence_groups(base_columns_for_v5)
V5_CHANGE_TRAIN, v5_change_source_audit = v5_change_feature_block(features_train, V5_GROUPS)
V5_CHANGE_TEST, _ = v5_change_feature_block(features_test, V5_GROUPS)
V5_FREQ_TRAIN, v5_freq_source_audit = v5_frequency_feature_block(features_train, V5_GROUPS)
V5_FREQ_TEST, _ = v5_frequency_feature_block(features_test, V5_GROUPS)
assert list(V5_CHANGE_TRAIN.columns) == list(V5_CHANGE_TEST.columns)
assert list(V5_FREQ_TRAIN.columns) == list(V5_FREQ_TEST.columns)
v5_change_audit = feature_block_audit(V5_CHANGE_TRAIN, v5_change_source_audit, OUT_DIR / "section07_v5_change_feature_audit.csv")
v5_freq_audit = feature_block_audit(V5_FREQ_TRAIN, v5_freq_source_audit, OUT_DIR / "section07_v5_frequency_feature_audit.csv")

features_train = pd.concat(
    [
        features_train.reset_index(drop=True),
        ADDITIVE_TRAIN.reset_index(drop=True),
        V5_CHANGE_TRAIN.reset_index(drop=True),
        V5_FREQ_TRAIN.reset_index(drop=True),
    ],
    axis=1,
)
features_test = pd.concat(
    [
        features_test.reset_index(drop=True),
        ADDITIVE_TEST.reset_index(drop=True),
        V5_CHANGE_TEST.reset_index(drop=True),
        V5_FREQ_TEST.reset_index(drop=True),
    ],
    axis=1,
)
features_train = features_train.loc[:, ~features_train.columns.duplicated()].copy()
features_test = features_test.loc[:, ~features_test.columns.duplicated()].copy()


def ranked_additive_features_for_export(label: str, topk: int):
    if topk <= 0:
        return [], "base_922_only"
    top_path = PATHS["section11_top_features"]
    if top_path.exists():
        top_df = pd.read_csv(top_path)
        needed = {"label", "feature", "feature_family", "mean_rank_score"}
        if needed.issubset(top_df.columns):
            candidates = (
                top_df[
                    top_df["label"].eq(label)
                    & ~top_df["feature_family"].eq("base_922")
                    & top_df["feature"].isin(features_train.columns)
                    & top_df["feature"].isin(features_test.columns)
                ]
                .sort_values("mean_rank_score", ascending=False)["feature"]
                .drop_duplicates()
                .head(topk)
                .tolist()
            )
            return candidates, f"section11_top_features_csv:{top_path}"
    fallback = [c for c in ADDITIVE_TRAIN.columns if c in features_train.columns and c in features_test.columns][:topk]
    return fallback, "fallback_v4_additive_column_order"


CURRENT07_FEATURES_BY_LABEL = {}
CURRENT07_POLICY_ROWS = []
for label, base_cols in ANCHOR_FEATURES_BY_LABEL.items():
    topk = int(SECTION11_EXPORT_ADDITIVE_TOPK_BY_LABEL.get(label, 0))
    additive_cols, additive_source = ranked_additive_features_for_export(label, topk)
    cols = list(dict.fromkeys([*base_cols, *additive_cols]))
    CURRENT07_FEATURES_BY_LABEL[label] = cols
    CURRENT07_POLICY_ROWS.append({
        "label": label,
        "base_feature_count": int(len(base_cols)),
        "requested_v4_additive_topk": topk,
        "resolved_v4_additive_count": int(len(additive_cols)),
        "current07_feature_count": int(len(cols)),
        "current07_feature_hash": feature_hash(cols),
        "v4_additive_source": additive_source,
        "v4_additive_features_json": json.dumps(additive_cols, ensure_ascii=False),
    })
CURRENT07_FEATURE_POLICY = pd.DataFrame(CURRENT07_POLICY_ROWS)


def v5_public_start_tail_split(y_df: pd.DataFrame, sample_df: pd.DataFrame):
    y_dates = pd.to_datetime(y_df["lifelog_date"])
    starts = sample_df.assign(lifelog_date=pd.to_datetime(sample_df["lifelog_date"])).groupby("subject_id")["lifelog_date"].min()
    row_starts = y_df["subject_id"].map(starts).fillna(pd.Timestamp.max)
    val_mask = y_dates >= row_starts
    val_idx = np.where(val_mask.to_numpy())[0]
    train_idx = np.where(~val_mask.to_numpy())[0]
    if len(val_idx) < 20:
        ordered = y_df.assign(_date=y_dates).sort_values(["subject_id", "_date"])
        val_parts = []
        for _, group in ordered.groupby("subject_id", sort=False):
            idx = list(group.index)
            val_parts.extend(idx[-max(1, int(np.ceil(len(idx) * 0.2))):])
        val_idx = np.asarray(sorted(set(val_parts)), dtype=int)
        train_idx = np.setdiff1d(np.arange(len(y_df)), val_idx)
    return [{"validation": "public_start_tail", "fold": 1, "train_idx": train_idx, "val_idx": val_idx}]


def v5_subject_time_tail_split(y_df: pd.DataFrame, holdout_ratio: float, name: str):
    ordered = y_df.assign(_date=pd.to_datetime(y_df["lifelog_date"])).sort_values(["subject_id", "_date"])
    val_parts = []
    for _, group in ordered.groupby("subject_id", sort=False):
        idx = list(group.index)
        val_parts.extend(idx[-max(1, int(np.ceil(len(idx) * holdout_ratio))):])
    val_idx = np.asarray(sorted(set(val_parts)), dtype=int)
    train_idx = np.setdiff1d(np.arange(len(y_df)), val_idx)
    return [{"validation": name, "fold": 1, "train_idx": train_idx, "val_idx": val_idx}]


def iter_v5_validation_splits(y_df: pd.DataFrame, sample_df: pd.DataFrame):
    for validation in SECTION07_V5_VALIDATIONS:
        if validation == "public_start_tail":
            yield from v5_public_start_tail_split(y_df, sample_df)
        elif validation == "subject_time_tail_25":
            yield from v5_subject_time_tail_split(y_df, 0.25, validation)
        elif validation == "subject_time_tail_35":
            yield from v5_subject_time_tail_split(y_df, 0.35, validation)
        else:
            raise ValueError(validation)


def assert_disjoint_indices(train_idx, val_idx, context: str) -> None:
    overlap = set(map(int, train_idx)) & set(map(int, val_idx))
    if overlap:
        raise AssertionError({"context": context, "n_overlap": len(overlap), "sample": sorted(overlap)[:10]})


def rank_v5_features_train_only(X_train: pd.DataFrame, y_train: np.ndarray, candidate_cols: list[str], topk: int) -> tuple[list[str], pd.DataFrame]:
    if topk <= 0 or not candidate_cols:
        return [], pd.DataFrame(columns=["feature", "rank_score", "abs_corr", "coverage", "n_unique"])
    rows = []
    y_s = pd.Series(y_train).astype(float).reset_index(drop=True)
    for col in candidate_cols:
        if col not in X_train.columns:
            continue
        s = pd.to_numeric(X_train[col], errors="coerce").reset_index(drop=True)
        coverage = float(s.notna().mean())
        n_unique = int(s.nunique(dropna=True))
        if coverage < 0.05 or n_unique <= 1:
            continue
        corr = float(abs(s.corr(y_s))) if s.notna().sum() > 2 else 0.0
        if not np.isfinite(corr):
            corr = 0.0
        rows.append({"feature": col, "abs_corr": corr, "coverage": coverage, "n_unique": n_unique})
    score_df = pd.DataFrame(rows)
    if score_df.empty:
        return [], score_df
    score_df["corr_rank"] = score_df["abs_corr"].rank(method="average", pct=True)
    score_df["coverage_rank"] = score_df["coverage"].rank(method="average", pct=True)
    score_df["rank_score"] = 0.85 * score_df["corr_rank"] + 0.15 * score_df["coverage_rank"]
    score_df = score_df.sort_values(["rank_score", "abs_corr", "coverage", "feature"], ascending=[False, False, False, True])
    return score_df["feature"].head(topk).tolist(), score_df


def v5_probe_lgbm_predict(X_train: pd.DataFrame, y_train: np.ndarray, X_val: pd.DataFrame, seed: int):
    import lightgbm as lgb
    params = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "n_estimators": SECTION07_V5_PROBE_N_ESTIMATORS,
        "learning_rate": 0.035,
        "max_depth": 4,
        "num_leaves": 16,
        "min_child_samples": 10,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.05,
        "reg_lambda": 3.0,
        "random_state": seed,
        "n_jobs": MODEL_CPU_THREADS,
        "verbosity": -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(sanitize_X(X_train), y_train)
    return np.clip(model.predict_proba(sanitize_X(X_val))[:, 1], CLIP_EPS, 1 - CLIP_EPS)


def v5_selection_score(label: str, mean_logloss: float, worst_logloss: float) -> float:
    if label in {"Q2", "Q3", "S4"}:
        return 0.6 * mean_logloss + 0.4 * worst_logloss
    return mean_logloss


V5_CHANGE_COLS = [c for c in V5_CHANGE_TRAIN.columns if c in features_train.columns and c in features_test.columns]
V5_FREQ_COLS = [c for c in V5_FREQ_TRAIN.columns if c in features_train.columns and c in features_test.columns]
v5_probe_rows = []
v5_selected_rows = []

if SECTION07_V5_ENABLE_PROBE:
    for label in LABELS:
        print(f"[SECTION07 V5 PROBE] {label}")
        y_all = train_y[label].astype(int).to_numpy()
        for split in iter_v5_validation_splits(train_y, sample):
            tr_idx = np.asarray(split["train_idx"], dtype=int)
            va_idx = np.asarray(split["val_idx"], dtype=int)
            assert_disjoint_indices(tr_idx, va_idx, f"v5_probe/{label}/{split['validation']}/{split['fold']}")
            y_tr, y_va = y_all[tr_idx], y_all[va_idx]
            base_cols = CURRENT07_FEATURES_BY_LABEL[label]
            change_topk = int(SECTION07_V5_CHANGE_TOPK_BY_LABEL.get(label, 0))
            freq_topk = int(SECTION07_V5_FREQ_TOPK_BY_LABEL.get(label, 0))
            change_cols, change_scores = rank_v5_features_train_only(features_train.iloc[tr_idx], y_tr, V5_CHANGE_COLS, change_topk)
            freq_cols, freq_scores = rank_v5_features_train_only(features_train.iloc[tr_idx], y_tr, V5_FREQ_COLS, freq_topk)
            for block_name, score_df in [("v5_change", change_scores), ("v5_frequency", freq_scores)]:
                if len(score_df):
                    tmp = score_df.head(max(change_topk, freq_topk, 1)).copy()
                    tmp.insert(0, "label", label)
                    tmp.insert(1, "validation", split["validation"])
                    tmp.insert(2, "fold", split["fold"])
                    tmp.insert(3, "block", block_name)
                    v5_selected_rows.extend(tmp.to_dict("records"))
            feature_sets = {
                "current_07_policy": base_cols,
                "current_07_plus_v5_change": list(dict.fromkeys([*base_cols, *change_cols])),
                "current_07_plus_v5_frequency": list(dict.fromkeys([*base_cols, *freq_cols])),
                "current_07_plus_v5_change_frequency": list(dict.fromkeys([*base_cols, *change_cols, *freq_cols])),
            }
            if label == "S1":
                feature_sets = {"current_07_policy": base_cols}
            if label == "S4":
                feature_sets = {
                    "current_07_policy": base_cols,
                    "current_07_plus_v5_change": list(dict.fromkeys([*base_cols, *change_cols])),
                }
            for feature_set, cols in feature_sets.items():
                missing = [c for c in cols if c not in features_train.columns or c not in features_test.columns]
                if missing:
                    raise KeyError({"label": label, "feature_set": feature_set, "missing": missing[:10]})
                seed = SECTION07_RANDOM_STATE + 507000 + 1000 * LABELS.index(label) + 101 * SECTION07_V5_VALIDATIONS.index(split["validation"])
                pred = v5_probe_lgbm_predict(features_train.iloc[tr_idx][cols], y_tr, features_train.iloc[va_idx][cols], seed)
                v5_probe_rows.append({
                    "label": label,
                    "validation": split["validation"],
                    "fold": int(split["fold"]),
                    "feature_set": feature_set,
                    "n_features": int(len(cols)),
                    "n_v5_change_features": int(sum(c in V5_CHANGE_COLS for c in cols)),
                    "n_v5_frequency_features": int(sum(c in V5_FREQ_COLS for c in cols)),
                    "val_logloss": binary_logloss(y_va, pred),
                    "val_pos_rate": float(np.mean(y_va)),
                })

v5_probe_metrics = pd.DataFrame(v5_probe_rows)
v5_selected_feature_scores = pd.DataFrame(v5_selected_rows)
if not v5_probe_metrics.empty:
    v5_probe_metrics.to_csv(OUT_DIR / "section07_v5_feature_probe_metrics.csv", index=False)
if not v5_selected_feature_scores.empty:
    v5_selected_feature_scores.to_csv(OUT_DIR / "section07_v5_selected_feature_scores.csv", index=False)

if not v5_probe_metrics.empty:
    v5_probe_summary = v5_probe_metrics.groupby(["label", "feature_set"], as_index=False).agg(
        mean_logloss=("val_logloss", "mean"),
        worst_validation_logloss=("val_logloss", "max"),
        mean_n_features=("n_features", "mean"),
        mean_n_v5_change_features=("n_v5_change_features", "mean"),
        mean_n_v5_frequency_features=("n_v5_frequency_features", "mean"),
    )
    v5_probe_summary["selection_score"] = v5_probe_summary.apply(
        lambda r: v5_selection_score(r["label"], r["mean_logloss"], r["worst_validation_logloss"]),
        axis=1,
    )
else:
    v5_probe_summary = pd.DataFrame(columns=["label", "feature_set", "mean_logloss", "worst_validation_logloss", "selection_score"])


def v5_final_cols_from_probe(label: str, block: str, topk: int) -> list[str]:
    if topk <= 0 or v5_selected_feature_scores.empty:
        return []
    part = v5_selected_feature_scores[
        v5_selected_feature_scores["label"].eq(label)
        & v5_selected_feature_scores["block"].eq(block)
    ].copy()
    if part.empty:
        candidates = V5_CHANGE_COLS if block == "v5_change" else V5_FREQ_COLS
        cols, _ = rank_v5_features_train_only(features_train, train_y[label].astype(int).to_numpy(), candidates, topk)
        return cols
    agg = part.groupby("feature", as_index=False).agg(
        mean_rank_score=("rank_score", "mean"),
        selected_count=("feature", "size"),
        mean_abs_corr=("abs_corr", "mean"),
    )
    return (
        agg.sort_values(["selected_count", "mean_rank_score", "mean_abs_corr", "feature"], ascending=[False, False, False, True])
        ["feature"].head(topk).tolist()
    )


def choose_v5_feature_set_by_label(summary: pd.DataFrame) -> tuple[pd.DataFrame, bool]:
    selected = []
    provisional = []
    for label in LABELS:
        cur = summary[(summary["label"].eq(label)) & (summary["feature_set"].eq("current_07_policy"))]
        if cur.empty:
            selected.append({"label": label, "selected_feature_set": "current_07_policy", "selected_reason": "missing_probe_current"})
            continue
        cur_row = cur.iloc[0]
        options = summary[summary["label"].eq(label)].copy()
        options = options.sort_values(["selection_score", "mean_logloss", "worst_validation_logloss", "feature_set"])
        best = options.iloc[0]
        reason = "best_probe_score"
        if label == "S1":
            best = cur_row
            reason = "s1_forced_base"
        elif label == "S4":
            strict = options[
                ~options["feature_set"].eq("current_07_policy")
                & (options["selection_score"] <= cur_row["selection_score"] - 0.001)
                & (options["mean_logloss"] <= cur_row["mean_logloss"])
                & (options["worst_validation_logloss"] <= cur_row["worst_validation_logloss"])
            ]
            if strict.empty:
                best = cur_row
                reason = "s4_strict_guard_base"
            else:
                best = strict.iloc[0]
                reason = "s4_strict_v5_win"
        elif label == "Q3":
            q3_win = options[
                ~options["feature_set"].eq("current_07_policy")
                & (
                    (options["mean_logloss"] < cur_row["mean_logloss"])
                    | (options["worst_validation_logloss"] < cur_row["worst_validation_logloss"])
                )
            ]
            if q3_win.empty:
                best = cur_row
                reason = "q3_no_mean_or_worst_gain"
            else:
                best = q3_win.iloc[0]
                reason = "q3_mean_or_worst_gain"
        else:
            if best["feature_set"] == "current_07_policy" or best["selection_score"] > cur_row["selection_score"] - 0.0005:
                best = cur_row
                reason = "v5_gain_too_small"
        selected.append({
            "label": label,
            "selected_feature_set": best["feature_set"],
            "selected_reason": reason,
            "selected_mean_logloss": float(best["mean_logloss"]),
            "selected_worst_validation_logloss": float(best["worst_validation_logloss"]),
            "selected_score": float(best["selection_score"]),
            "current_mean_logloss": float(cur_row["mean_logloss"]),
            "current_worst_validation_logloss": float(cur_row["worst_validation_logloss"]),
            "current_score": float(cur_row["selection_score"]),
            "score_gain_vs_current": float(cur_row["selection_score"] - best["selection_score"]),
        })
        provisional.append((label, best["feature_set"], cur_row, best))
    selected_df = pd.DataFrame(selected)
    focus = selected_df[selected_df["label"].isin(["Q2", "Q3", "S3"])].copy()
    focus_improved = int((focus["score_gain_vs_current"] > 0).sum()) if len(focus) else 0
    current_overall = float(selected_df["current_mean_logloss"].mean()) if len(selected_df) else np.inf
    selected_overall = float(selected_df["selected_mean_logloss"].mean()) if len(selected_df) else np.inf
    q3 = selected_df[selected_df["label"].eq("Q3")]
    q3_improved = bool(len(q3) and q3["selected_feature_set"].iloc[0] != "current_07_policy" and (
        q3["selected_mean_logloss"].iloc[0] < q3["current_mean_logloss"].iloc[0]
        or q3["selected_worst_validation_logloss"].iloc[0] < q3["current_worst_validation_logloss"].iloc[0]
    ))
    accepted = bool(q3_improved and focus_improved >= 2 and selected_overall <= current_overall)
    selected_df["v5_full_acceptance"] = accepted
    selected_df["focus_improved_count"] = focus_improved
    selected_df["current_overall_mean_logloss"] = current_overall
    selected_df["selected_overall_mean_logloss"] = selected_overall
    if SECTION07_V5_REQUIRE_ACCEPTANCE_FOR_EXPORT and not accepted:
        keep_q3 = q3_improved
        mask_revert = ~selected_df["label"].eq("Q3") if keep_q3 else pd.Series(True, index=selected_df.index)
        selected_df.loc[mask_revert, "selected_feature_set"] = "current_07_policy"
        selected_df.loc[mask_revert, "selected_reason"] = "global_v5_acceptance_failed_revert_to_current"
        selected_df.loc[mask_revert, "selected_mean_logloss"] = selected_df.loc[mask_revert, "current_mean_logloss"]
        selected_df.loc[mask_revert, "selected_worst_validation_logloss"] = selected_df.loc[mask_revert, "current_worst_validation_logloss"]
        selected_df.loc[mask_revert, "selected_score"] = selected_df.loc[mask_revert, "current_score"]
        selected_df.loc[mask_revert, "score_gain_vs_current"] = 0.0
    return selected_df, accepted


if not v5_probe_summary.empty:
    v5_labelwise_policy, v5_full_accepted = choose_v5_feature_set_by_label(v5_probe_summary)
else:
    v5_labelwise_policy = pd.DataFrame([
        {"label": label, "selected_feature_set": "current_07_policy", "selected_reason": "probe_disabled_or_empty"}
        for label in LABELS
    ])
    v5_full_accepted = False

v5_probe_summary.to_csv(OUT_DIR / "section07_v5_feature_probe_summary.csv", index=False)
v5_labelwise_policy.to_csv(OUT_DIR / "section07_v5_labelwise_feature_policy.csv", index=False)


EXPORT_FEATURES_BY_LABEL = {}
EXPORT_FEATURE_POLICY_ROWS = []
for label, base_cols in ANCHOR_FEATURES_BY_LABEL.items():
    current_cols = CURRENT07_FEATURES_BY_LABEL[label]
    selected_feature_set = v5_labelwise_policy[v5_labelwise_policy["label"].eq(label)]["selected_feature_set"]
    selected_feature_set = selected_feature_set.iloc[0] if len(selected_feature_set) else "current_07_policy"
    change_topk = int(SECTION07_V5_CHANGE_TOPK_BY_LABEL.get(label, 0))
    freq_topk = int(SECTION07_V5_FREQ_TOPK_BY_LABEL.get(label, 0))
    v5_change_cols = v5_final_cols_from_probe(label, "v5_change", change_topk) if "v5_change" in selected_feature_set else []
    v5_freq_cols = v5_final_cols_from_probe(label, "v5_frequency", freq_topk) if "v5_frequency" in selected_feature_set else []
    cols = list(dict.fromkeys([*current_cols, *v5_change_cols, *v5_freq_cols]))
    missing_cols = [c for c in cols if c not in features_train.columns or c not in features_test.columns]
    if missing_cols:
        raise KeyError({"label": label, "missing_export_features": missing_cols[:20], "n_missing": len(missing_cols)})
    EXPORT_FEATURES_BY_LABEL[label] = cols
    current_row = CURRENT07_FEATURE_POLICY[CURRENT07_FEATURE_POLICY["label"].eq(label)].iloc[0].to_dict()
    selected_row = v5_labelwise_policy[v5_labelwise_policy["label"].eq(label)]
    selected_payload = selected_row.iloc[0].to_dict() if len(selected_row) else {}
    EXPORT_FEATURE_POLICY_ROWS.append({
        "label": label,
        "base_feature_count": int(len(base_cols)),
        "requested_additive_topk": int(current_row["requested_v4_additive_topk"]),
        "resolved_additive_count": int(current_row["resolved_v4_additive_count"]),
        "selected_feature_set": selected_feature_set,
        "selected_reason": selected_payload.get("selected_reason"),
        "requested_v5_change_topk": change_topk,
        "resolved_v5_change_count": int(len(v5_change_cols)),
        "requested_v5_frequency_topk": freq_topk,
        "resolved_v5_frequency_count": int(len(v5_freq_cols)),
        "v5_full_acceptance": bool(v5_full_accepted),
        "selected_score": selected_payload.get("selected_score"),
        "current_score": selected_payload.get("current_score"),
        "score_gain_vs_current": selected_payload.get("score_gain_vs_current"),
        "export_feature_count": int(len(cols)),
        "export_feature_hash": feature_hash(cols),
        "additive_source": current_row["v4_additive_source"],
        "additive_features_json": current_row["v4_additive_features_json"],
        "v5_change_features_json": json.dumps(v5_change_cols, ensure_ascii=False),
        "v5_frequency_features_json": json.dumps(v5_freq_cols, ensure_ascii=False),
    })

EXPORT_FEATURE_POLICY = pd.DataFrame(EXPORT_FEATURE_POLICY_ROWS)
EXPORT_FEATURE_POLICY.to_csv(OUT_DIR / "section07_section4_retrain_feature_policy.csv", index=False)

feature_manifest = {
    "created_at": datetime.now().isoformat(),
    "feature_source": FEATURE_SOURCE,
    "export_feature_source": EXPORT_FEATURE_SOURCE,
    "base_policy": "preserve saved 922 anchor features per label",
    "additive_policy": SECTION11_EXPORT_ADDITIVE_TOPK_BY_LABEL,
    "v5_change_topk_policy": SECTION07_V5_CHANGE_TOPK_BY_LABEL,
    "v5_frequency_topk_policy": SECTION07_V5_FREQ_TOPK_BY_LABEL,
    "v5_validations": SECTION07_V5_VALIDATIONS,
    "v5_full_acceptance": bool(v5_full_accepted),
    "feature_counts_by_label": {label: len(cols) for label, cols in EXPORT_FEATURES_BY_LABEL.items()},
    "feature_hash_by_label": {label: feature_hash(cols) for label, cols in EXPORT_FEATURES_BY_LABEL.items()},
    "additive_feature_count": int(ADDITIVE_TRAIN.shape[1]),
    "additive_feature_hash": feature_hash(list(ADDITIVE_TRAIN.columns)),
    "v5_change_feature_count": int(V5_CHANGE_TRAIN.shape[1]),
    "v5_change_feature_hash": feature_hash(list(V5_CHANGE_TRAIN.columns)),
    "v5_frequency_feature_count": int(V5_FREQ_TRAIN.shape[1]),
    "v5_frequency_feature_hash": feature_hash(list(V5_FREQ_TRAIN.columns)),
    "policy_path": str(OUT_DIR / "section07_section4_retrain_feature_policy.csv"),
    "v5_change_audit_path": str(OUT_DIR / "section07_v5_change_feature_audit.csv"),
    "v5_frequency_audit_path": str(OUT_DIR / "section07_v5_frequency_feature_audit.csv"),
    "v5_probe_metrics_path": str(OUT_DIR / "section07_v5_feature_probe_metrics.csv"),
    "v5_probe_summary_path": str(OUT_DIR / "section07_v5_feature_probe_summary.csv"),
    "v5_labelwise_policy_path": str(OUT_DIR / "section07_v5_labelwise_feature_policy.csv"),
    "no_submission_or_prediction_input": True,
}
write_json(OUT_DIR / "section07_section4_retrain_feature_manifest.json", feature_manifest)
write_json(OUT_DIR / "section07_v5_feature_manifest.json", feature_manifest)

display(EXPORT_FEATURE_POLICY)
display(v5_probe_summary.sort_values(["label", "selection_score", "feature_set"]) if len(v5_probe_summary) else v5_probe_summary)
display(v5_labelwise_policy)
print("EXPORT_FEATURE_COUNTS:", feature_manifest["feature_counts_by_label"])
print("EXPORT_FEATURE_SOURCE:", EXPORT_FEATURE_SOURCE)
print("V5_FULL_ACCEPTANCE:", v5_full_accepted)


## 4. Section 4 Model Utility Replay

In [ ]:

def get_lgbm_classifier(params=None, random_state: int = 42):
    import lightgbm as lgb
    base = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "random_state": random_state,
        "n_jobs": MODEL_CPU_THREADS,
        "verbosity": -1,
        "subsample_freq": 1,
    }
    if params:
        base.update(params)
    return lgb.LGBMClassifier(**base)


def get_catboost_classifier(params=None, random_state: int = 42):
    from catboost import CatBoostClassifier
    base = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "random_seed": random_state,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": MODEL_CPU_THREADS,
    }
    if params:
        base.update(params)
    return CatBoostClassifier(**base)


def optimize_lgbm_params(X: pd.DataFrame, y: np.ndarray, n_trials: int, random_state: int):
    import lightgbm as lgb
    import optuna
    inner_tr, inner_va = make_inner_es_split(y, random_state)
    X_tr, X_va = X.iloc[inner_tr], X.iloc[inner_va]
    y_tr, y_va = y[inner_tr], y[inner_va]
    trial_rows = []

    def objective(trial):
        max_depth = trial.suggest_int("max_depth", 3, 8)
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
            "max_depth": max_depth,
            "num_leaves": trial.suggest_int("num_leaves", 8, min(2 ** max_depth, 96)),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 2.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
        }
        model = get_lgbm_classifier(params, random_state=random_state)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(period=0)],
        )
        pred = model.predict_proba(X_va)[:, 1]
        score = binary_logloss(y_va, pred)
        trial_rows.append({"base_model": "lgbm", "trial": trial.number, "logloss": score, **params})
        return score

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, n_jobs=OPTUNA_N_JOBS, show_progress_bar=OPTUNA_SHOW_PROGRESS_BAR)
    return dict(study.best_params), pd.DataFrame(trial_rows)


def optimize_catboost_params(X: pd.DataFrame, y: np.ndarray, n_trials: int, random_state: int):
    import optuna
    inner_tr, inner_va = make_inner_es_split(y, random_state)
    X_tr, X_va = X.iloc[inner_tr], X.iloc[inner_va]
    y_tr, y_va = y[inner_tr], y[inner_va]
    trial_rows = []

    def objective(trial):
        params = {
            "iterations": trial.suggest_int("iterations", 300, 1500),
            "depth": trial.suggest_int("depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.5, 10.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.1, 3.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        }
        model = get_catboost_classifier(params, random_state=random_state)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False)
        pred = model.predict_proba(X_va)[:, 1]
        score = binary_logloss(y_va, pred)
        trial_rows.append({"base_model": "catboost", "trial": trial.number, "logloss": score, **params})
        return score

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, n_jobs=OPTUNA_N_JOBS, show_progress_bar=OPTUNA_SHOW_PROGRESS_BAR)
    return dict(study.best_params), pd.DataFrame(trial_rows)


def fit_base_model_for_test(base_model: str, X_train: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, seed: int):
    import lightgbm as lgb
    if base_model == "lgbm":
        params, trials = optimize_lgbm_params(X_train, y, SECTION07_TRIALS, seed)
        inner_tr, inner_es = make_inner_es_split(y, seed + 1000)
        model = get_lgbm_classifier(params, random_state=seed + 2000)
        model.fit(
            X_train.iloc[inner_tr],
            y[inner_tr],
            eval_set=[(X_train.iloc[inner_es], y[inner_es])],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(period=0)],
        )
        return np.clip(model.predict_proba(X_test)[:, 1], CLIP_EPS, 1 - CLIP_EPS), params, trials
    if base_model == "catboost":
        params, trials = optimize_catboost_params(X_train, y, SECTION07_TRIALS, seed)
        inner_tr, inner_es = make_inner_es_split(y, seed + 1000)
        model = get_catboost_classifier(params, random_state=seed + 2000)
        model.fit(
            X_train.iloc[inner_tr],
            y[inner_tr],
            eval_set=(X_train.iloc[inner_es], y[inner_es]),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )
        return np.clip(model.predict_proba(X_test)[:, 1], CLIP_EPS, 1 - CLIP_EPS), params, trials
    raise ValueError(base_model)


## 5. Seed-Ensemble Model Optuna Export

In [ ]:

baseline_submission = sample.copy()
seed_pack_submissions = {}
baseline_label_rows = []
baseline_param_payload = {}
all_trial_frames = []
prediction_long_frames = []


def run_lgbm_catboost_pack(label: str, seed_offset: int, pack_name: str):
    cols = EXPORT_FEATURES_BY_LABEL[label]
    X_train = sanitize_X(features_train[cols].reset_index(drop=True))
    X_test = sanitize_X(features_test[cols].reset_index(drop=True))
    y = train_y[label].astype(int).to_numpy()
    base_preds, base_params, label_trials = {}, {}, []
    for i, base_model in enumerate(["lgbm", "catboost"]):
        seed = SECTION07_RANDOM_STATE + 61000 + seed_offset + 1000 * LABELS.index(label) + 31 * i
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {pack_name} {label}/{base_model}: n_trials={SECTION07_TRIALS}, seed={seed}")
        pred, params, trials = fit_base_model_for_test(base_model, X_train, y, X_test, seed)
        base_preds[base_model] = pred
        base_params[base_model] = params
        if len(trials):
            trials = trials.copy()
            trials.insert(0, "label", label)
            trials.insert(1, "pack_name", pack_name)
            trials.insert(2, "seed_offset", seed_offset)
            trials.insert(3, "n_features", len(cols))
            label_trials.append(trials)
            all_trial_frames.append(trials)
    pred = np.clip(np.mean([base_preds["lgbm"], base_preds["catboost"]], axis=0), CLIP_EPS, 1 - CLIP_EPS)
    return pred, base_preds, base_params, label_trials


for seed_offset in SECTION07_SEED_ENSEMBLE_OFFSETS:
    pack_name = "baseline_section4_retrain" if seed_offset == 0 else f"seed_ensemble_offset_{seed_offset}"
    pack_submission = sample.copy()
    seed_pack_submissions[pack_name] = pack_submission
    for label_idx, label in enumerate(LABELS_TO_RUN, start=1):
        print("\n" + "=" * 88)
        print(f"[{pack_name}] LABEL {label_idx}/{len(LABELS_TO_RUN)} {label}")
        print("=" * 88)
        pred, base_preds, base_params, label_trials = run_lgbm_catboost_pack(label, seed_offset, pack_name)
        pack_submission[label] = pred
        if seed_offset == 0:
            baseline_submission[label] = pred
            policy_row = EXPORT_FEATURE_POLICY[EXPORT_FEATURE_POLICY["label"].eq(label)].iloc[0].to_dict()
            baseline_label_rows.append({
                "label": label,
                "model": MODEL_NAME,
                "normalization": NORMALIZATION,
                "feature_source": EXPORT_FEATURE_SOURCE,
                "base_feature_count": int(policy_row["base_feature_count"]),
                "resolved_additive_count": int(policy_row["resolved_additive_count"]),
                "requested_additive_topk": int(policy_row["requested_additive_topk"]),
                "additive_source": policy_row["additive_source"],
                "blend_method": BASELINE_INTERNAL_BLEND_METHOD,
                "n_trials": SECTION07_TRIALS,
                "n_features": int(policy_row["export_feature_count"]),
                "feature_hash": policy_row["export_feature_hash"],
                "lgbm_best_inner_logloss": float(pd.concat(label_trials).query("base_model == 'lgbm'")["logloss"].min()) if label_trials else np.nan,
                "catboost_best_inner_logloss": float(pd.concat(label_trials).query("base_model == 'catboost'")["logloss"].min()) if label_trials else np.nan,
            })
            baseline_param_payload[label] = {
                "features": EXPORT_FEATURES_BY_LABEL[label],
                "feature_hash": policy_row["export_feature_hash"],
                "base_model_params": base_params,
                "model": MODEL_NAME,
                "normalization": NORMALIZATION,
                "feature_source": EXPORT_FEATURE_SOURCE,
                "feature_policy": policy_row,
                "blend_method": BASELINE_INTERNAL_BLEND_METHOD,
                "n_trials": SECTION07_TRIALS,
            }
            prediction_long_frames.append(pd.DataFrame({
                "subject_id": sample["subject_id"],
                "lifelog_date": sample["lifelog_date"],
                "label": label,
                "candidate_name": "baseline_section4_retrain",
                "lgbm_pred": base_preds["lgbm"],
                "catboost_pred": base_preds["catboost"],
                "blend_pred": pred,
            }))

for label in LABELS:
    if label not in LABELS_TO_RUN:
        baseline_submission[label] = np.clip(sample[label].astype(float), CLIP_EPS, 1 - CLIP_EPS)
        for pack_submission in seed_pack_submissions.values():
            pack_submission[label] = baseline_submission[label]
for sub in [baseline_submission, *seed_pack_submissions.values()]:
    assert len(sub) == len(sample)
    assert list(sub.columns) == list(sample.columns)
    for label in LABELS:
        sub[label] = np.clip(sub[label].astype(float), CLIP_EPS, 1 - CLIP_EPS)

baseline_selection_df = pd.DataFrame(baseline_label_rows)
baseline_selection_df.to_csv(OUT_DIR / "section07_section4_retrain_selection.csv", index=False)
write_json(OUT_DIR / "section07_section4_retrain_params.json", baseline_param_payload)
if all_trial_frames:
    pd.concat(all_trial_frames, ignore_index=True).to_csv(OUT_DIR / "section07_section4_retrain_trials.csv", index=False)

def make_candidate(name: str) -> pd.DataFrame:
    out = baseline_submission.copy()
    if name == "baseline_seed_ensemble":
        for label in LABELS_TO_RUN:
            preds = {pack: sub[label].to_numpy() for pack, sub in seed_pack_submissions.items()}
            out[label] = logit_blend(preds)
        return out
    raise ValueError(name)


candidate_names = [
    "baseline_seed_ensemble",
]

candidate_rows = []
drift_rows = []
for name in candidate_names:
    candidate = make_candidate(name)
    assert len(candidate) == len(sample)
    assert list(candidate.columns) == list(sample.columns)
    for label in LABELS:
        candidate[label] = np.clip(candidate[label].astype(float), CLIP_EPS, 1 - CLIP_EPS)
        assert candidate[label].between(CLIP_EPS, 1 - CLIP_EPS).all(), label
    path = SUBMISSION_DIR / f"section07_candidate_{name}_{SECTION07_TIMESTAMP}.csv"
    metadata_path = SUBMISSION_DIR / f"section07_candidate_{name}_{SECTION07_TIMESTAMP}_metadata.json"
    candidate.to_csv(path, index=False)
    row = {
        "candidate_name": name,
        "submission_path": str(path),
        "metadata_path": str(metadata_path),
        "uses_seed_ensemble": bool("seed_ensemble" in name),
        "baseline_internal_blend_method": BASELINE_INTERNAL_BLEND_METHOD,
        "candidate_blend_method": CANDIDATE_BLEND_METHOD,
        "seed_offsets": json.dumps(SECTION07_SEED_ENSEMBLE_OFFSETS),
        "n_trials": SECTION07_TRIALS,
        "n_labels": len(LABELS_TO_RUN),
    }
    candidate_rows.append(row)
    drift_rows.append(prediction_stats(candidate, name, baseline_submission))
    write_json(metadata_path, {
        "created_at": datetime.now().isoformat(),
        **row,
        "no_existing_submission_or_prediction_input": True,
        "allowed_input_audit": str(OUT_DIR / "section07_allowed_input_audit.csv"),
        "feature_policy_path": str(OUT_DIR / "section07_section4_retrain_feature_policy.csv"),
        "baseline_params_path": str(OUT_DIR / "section07_section4_retrain_params.json"),
    })

scoreboard = pd.DataFrame(candidate_rows)
drift_audit = pd.DataFrame(drift_rows)
scoreboard.to_csv(OUT_DIR / "section07_candidate_scoreboard.csv", index=False)
drift_audit.to_csv(OUT_DIR / "section07_prediction_drift_audit.csv", index=False)
if prediction_long_frames:
    pd.concat(prediction_long_frames, ignore_index=True).to_parquet(OUT_DIR / "section07_baseline_predictions_long.parquet", index=False)

write_json(OUT_DIR / "section07_model_optuna_seed_ensemble_config.json", {
    "created_at": datetime.now().isoformat(),
    "mode": SECTION07_MODE,
    "labels_to_run": LABELS_TO_RUN,
    "trials": SECTION07_TRIALS,
    "model_cpu_threads": MODEL_CPU_THREADS,
    "optuna_n_jobs": OPTUNA_N_JOBS,
    "baseline_model": MODEL_NAME,
    "baseline_internal_blend_method": BASELINE_INTERNAL_BLEND_METHOD,
    "candidate_blend_method": CANDIDATE_BLEND_METHOD,
    "seed_ensemble_offsets": SECTION07_SEED_ENSEMBLE_OFFSETS,
    "model_optimization_scope": "mix_lgbm_catboost_seed_ensemble_only",
    "no_existing_submission_or_prediction_input": True,
})

report_path = REPORT_DIR / "section07_model_optuna_seed_ensemble_summary.md"
report_lines = [
    "# Section 07 Section4-Retrain Seed-Ensemble Model Optimization Summary",
    "",
    f"- created_at: `{datetime.now().isoformat()}`",
    f"- mode: `{SECTION07_MODE}`",
    f"- trials: `{SECTION07_TRIALS}`",
    f"- labels: `{LABELS_TO_RUN}`",
    f"- baseline_model: `{MODEL_NAME}`",
    f"- baseline_internal_blend_method: `{BASELINE_INTERNAL_BLEND_METHOD}`",
    f"- candidate_blend_method: `{CANDIDATE_BLEND_METHOD}`",
    f"- seed_offsets: `{SECTION07_SEED_ENSEMBLE_OFFSETS}`",
    "- exported candidates: `baseline_seed_ensemble` only.",
    "- removed model branches: Q2/Q3 XGB challenger, S4 CatBoost-heavy challenger, conservative bottleneck blends.",
    "- input policy: no existing submission/test prediction file is read.",
    f"- v5_full_acceptance: `{v5_full_accepted}`",
    "",
    "## Feature Policy",
    "",
    EXPORT_FEATURE_POLICY.to_string(index=False),
    "",
    "## V5 Labelwise Feature Policy",
    "",
    v5_labelwise_policy.to_string(index=False),
    "",
    "## V5 Feature Probe Summary",
    "",
    v5_probe_summary.sort_values(["label", "selection_score", "feature_set"]).to_string(index=False) if len(v5_probe_summary) else "(empty)",
    "",
    "## Candidate Scoreboard",
    "",
    scoreboard.to_string(index=False),
    "",
    "## Artifacts",
    "",
    f"- allowed_input_audit: `{OUT_DIR / 'section07_allowed_input_audit.csv'}`",
    f"- feature_policy: `{OUT_DIR / 'section07_section4_retrain_feature_policy.csv'}`",
    f"- v5_change_audit: `{OUT_DIR / 'section07_v5_change_feature_audit.csv'}`",
    f"- v5_frequency_audit: `{OUT_DIR / 'section07_v5_frequency_feature_audit.csv'}`",
    f"- v5_probe_metrics: `{OUT_DIR / 'section07_v5_feature_probe_metrics.csv'}`",
    f"- v5_probe_summary: `{OUT_DIR / 'section07_v5_feature_probe_summary.csv'}`",
    f"- v5_labelwise_policy: `{OUT_DIR / 'section07_v5_labelwise_feature_policy.csv'}`",
    f"- baseline_params: `{OUT_DIR / 'section07_section4_retrain_params.json'}`",
    f"- scoreboard: `{OUT_DIR / 'section07_candidate_scoreboard.csv'}`",
    f"- drift_audit: `{OUT_DIR / 'section07_prediction_drift_audit.csv'}`",
]
report_path.write_text("\n".join(report_lines), encoding="utf-8")

display(scoreboard)
display(drift_audit)
print("SECTION07_SCOREBOARD:", OUT_DIR / "section07_candidate_scoreboard.csv")
print("SECTION07_REPORT:", report_path)
print("SECTION07_SUBMISSION_DIR:", SUBMISSION_DIR)
